# 🔄 Alapvető Ügynök Munkafolyamatok a Microsoft Foundry használatával (Python)

## 📋 Munkafolyamat-orchesztrációs Oktatóanyag

Ez a jegyzetfüzet bemutatja a Microsoft Agent Framework erőteljes **Workflow Builder** képességeit. Tanulja meg, hogyan hozhat létre kifinomult, többlépcsős ügynök munkafolyamatokat, amelyek képesek összetett üzleti folyamatok kezelésére és több MI művelet zökkenőmentes összehangolására.

> **Áttelepítési megjegyzés:** Ez a példa korábban a GitHub Models-re hivatkozott. A GitHub Models elavult (2026 júliusában megszűnik), ezért most a **Microsoft Foundry**-t használja a `FoundryChatClient`-en keresztül, amely az Azure OpenAI **Responses API**-ját célozza meg.

## 🎯 Tanulási célok

### 🏗️ **Munkafolyamat Felépítése**
- **Workflow Builder**: Összetett többlépcsős folyamatok tervezése és irányítása
- **Eseményvezérelt Végrehajtás**: Munkafolyamat események és állapotváltozások kezelése
- **Látványos Munkafolyamat Tervezés**: Munkafolyamat struktúrák létrehozása és megjelenítése
- **Microsoft Foundry Integráció**: MI modellek kihasználása a munkafolyamatokban

### 🔄 **Folyamat-kezelés**
- **Szekvenciális Műveletek**: Több ügynöki feladat logikus sorrendbe fűzése
- **Feltételes Logika**: Döntési pontok és ágak megvalósítása a munkafolyamatokban
- **Hibakezelés**: Robosztus hibahelyreállítás és munkafolyamat ellenállás
- **Állapotkezelés**: A munkafolyamat végrehajtási állapotának követése és kezelése

### 📊 **Vállalati Munkafolyamat Minták**
- **Üzleti Folyamat Automatizálás**: Összetett szervezeti munkafolyamatok automatizálása
- **Több Ügynök Koordinációja**: Több specializált ügynök összehangolása
- **Skálázható Végrehajtás**: Munkafolyamatok tervezése vállalati szintű műveletekhez
- **Monitorozás és Megfigyelhetőség**: A munkafolyamat teljesítményének és eredményeinek követése

## ⚙️ Előfeltételek & Beállítás

### 📦 **Szükséges Függőségek**

Telepítse az Agent Framework-öt munkafolyamat képességekkel:

```bash
pip install agent-framework -U
```

### 🔑 **Microsoft Foundry Konfiguráció**

Jelentkezzen be az Azure CLI-vel (`az login`), hogy az `AzureCliCredential` hitelesíteni tudjon, majd állítsa be a Microsoft Foundry projekt adatait.

**Környezeti beállítás (.env fájl):**
```env
AZURE_AI_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-5-mini
```

### 🏢 **Vállalati Felhasználási Esetek**

**Üzleti folyamat példák:**
- **Ügyfélregisztráció**: Többlépcsős ellenőrzési és beállítási munkafolyamatok
- **Tartalom-folyamat**: Automatikus tartalomkészítés, ellenőrzés és kiadás
- **Adatfeldolgozás**: ETL munkafolyamatok MI-alapú átalakítással
- **Minőségbiztosítás**: Automatikus tesztelési és érvényesítési folyamatok

**A munkafolyamat előnyei:**
- 🎯 **Megbízhatóság**: Determinisztikus végrehajtás hibahelyreállítással
- 📈 **Skálázhatóság**: Nagy volumenű folyamatautomatizálás kezelése
- 🔍 **Megfigyelhetőség**: Teljes audit nyomvonalak és monitorozás
- 🔧 **Karbantarthatóság**: Látványos tervezés és moduláris komponensek

## 🎨 Munkafolyamat Tervezési Minták

### Alapvető Munkafolyamat Felépítés
```mermaid
graph TD
    A[Indítás] --> B[Ügynök Feladat 1]
    B --> C{Döntési Pont}
    C -->|Siker| D[Ügynök Feladat 2]
    C -->|Sikertelenség| E[Hibakezelő]
    D --> F[Végállomás]
    E --> F
```

**Kulcsfontosságú elemek:**
- **WorkflowBuilder**: Fő orchesztrációs motor
- **WorkflowEvent**: Eseménykezelés és kommunikáció
- **WorkflowViz**: Munkafolyamat vizuális ábrázolása és hibakeresés

Építsük meg az első intelligens munkafolyamatodat! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
# Core components for building sophisticated agent workflows
from agent_framework import WorkflowBuilder, WorkflowEvent, WorkflowViz
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential


In [ ]:
# 📦 Import Environment and System Utilities
# Essential libraries for configuration and environment management

import os                      # 🔧 Environment variable access
from dotenv import load_dotenv # 📁 Secure configuration loading

In [ ]:
# 🔧 Initialize Environment Configuration
# Load Microsoft Foundry project settings from .env file
load_dotenv()


In [ ]:
# Configure the Microsoft Foundry client with keyless authentication.
# FoundryChatClient targets the Azure OpenAI Responses API.
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


In [ ]:
REVIEWER_NAME = "Concierge"
REVIEWER_INSTRUCTIONS = """
    You are an are hotel concierge who has opinions about providing the most local and authentic experiences for travelers.
    The goal is to determine if the front desk travel agent has recommended the best non-touristy experience for a traveler.
    If so, state that it is approved.
    If not, provide insight on how to refine the recommendation without using a specific example. 
    """

In [ ]:
FRONTDESK_NAME = "FrontDesk"
FRONTDESK_INSTRUCTIONS = """
    You are a Front Desk Travel Agent with ten years of experience and are known for brevity as you deal with many customers.
    The goal is to provide the best activities and locations for a traveler to visit.
    Only provide a single recommendation per response.
    You're laser focused on the goal at hand.
    Don't waste time with chit chat.
    Consider suggestions when refining an idea.
    """

In [ ]:
reviewer_agent = provider.as_agent(
    name=REVIEWER_NAME,
    instructions=REVIEWER_INSTRUCTIONS,
)

front_desk_agent = provider.as_agent(
    name=FRONTDESK_NAME,
    instructions=FRONTDESK_INSTRUCTIONS,
)


In [ ]:
workflow = (
    WorkflowBuilder(start_executor=front_desk_agent)
    .add_edge(front_desk_agent, reviewer_agent)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
# Display the exported workflow SVG inline in the notebook

from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")


In [ ]:
# Workflow.run_stream is no longer part of the public API; the current Workflow
# returns a results object whose `get_outputs()` produces the AgentResponse from
# each output executor. The reviewer (last stage) is the only output here.
events = await workflow.run("I would like to go to Paris.")
outputs = events.get_outputs()
result = outputs[0].text if outputs else ""

In [ ]:
result.replace("None", "")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordítási szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár az pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Fontos információk esetén professzionális emberi fordítást javasolunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely ebből a fordításból ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
